# 가설 검증

가설 1
- H0 : gate 40의 D1 retention은 gate 30의 D1 retention이 같다.
- H1 : gate 40의 D1 retention이 gate 30의 D1 retention보다 크다.

가설 2
- H0 : gate 40의 D7 retention은 gate 30의 D7 retention이 같다.
- H1 : gate 40의 D7 retention이 gate 30의 D7 retention보다 크다.

In [59]:
import pandas as pd
import math
from scipy.stats import norm

In [14]:
df = pd.read_csv('../data/cookie_cats.csv')
df.sample(5)

,userid,version,sum_gamerounds,retention_1,retention_7
11945,1319560,gate_40,350,True,True
70382,7804147,gate_40,124,True,False
62019,6869714,gate_30,2,False,False
47527,5273848,gate_40,22,True,False
80050,8860750,gate_40,30,False,False


## retention 계산
- total retention = dn cnt / d0 cnt
- retention(group) = dn_group cnt / d0_group cnt

In [37]:
# 전체 수
d0_30 = df.groupby('version').userid.count().loc['gate_30']
d0_40 = df.groupby('version').userid.count().loc['gate_40']

print(d0_30)
print(d0_40)

44700
45489


In [45]:
d1_30 = df.groupby('version').retention_1.sum().loc['gate_30']
d1_40 = df.groupby('version').retention_1.sum().loc['gate_40']
d7_30 = df.groupby('version').retention_7.sum().loc['gate_30']
d7_40 = df.groupby('version').retention_7.sum().loc['gate_40']
print(d1_30, d1_40, d7_30, d7_40)

20034 20119 8502 8279


In [42]:
r_d1_30 = df[df.version == 'gate_30']['retention_1'].mean()   # gate_30 d1 retention
r_d7_30 = df[df.version == 'gate_30']['retention_7'].mean()   # gate_30 d7 retention
r_d1_40 = df[df.version == 'gate_40']['retention_1'].mean()   # gate_40 d1 retention
r_d7_40 = df[df.version == 'gate_40']['retention_7'].mean()   # gate_40 d7 retention

print(r_d1_30, r_d7_30, r_d1_40, r_d7_40)

0.4481879194630872 0.19020134228187918 0.44228274967574577 0.18200004396667327


## z-test
- 비율과 분포가 대부분 동일했기에 pooled로 계산.

In [81]:
# pooled proportion 함수
def pooled_proportion(a, b, t1, t2):
    '''
    a : group1 day cnt
    b : group2 day cnt
    t1: group1 total cnt
    t2: group2 total cnt
    '''
    return (a + b) / (t1 + t2)


# 표준오차 함수
def se_func(p_pool, t1, t2):
    '''
    p_pool: pooled proportion
    t1: group1 total cnt
    t2: group2 total cnt
    '''
    return math.sqrt(p_pool * (1 - p_pool) * ((1 / t1) + (1 / t2)))


# z-score 함수
def z_score(r1, r2, se):
    '''
    r1 : 대립가설 기준 더 크다는 걸 검증하고 싶은 값.
    r2 : 이외 값
    se : 표준오차
    '''
    return (r1 - r2) / se


# 결과 출력 
def print_result(day, r1, r2, pp, se, zs):
    '''
    r1: retention1
    r2: retention2
    pp: pooled proportion
    se: se
    zs: z-score
    '''

    p_value = 1 - norm.cdf(zs)

    print(f"group30 {day} retention: {r2:.4f}, group40 {day} retention: {r1:.4f}")
    print(f"pooled p: {pp:.4f}")
    print(f"SE: {se:.6f}")
    print(f"z-score: {zs:.4f}")

    if zs < 0:
        print("가설 방향과 반대 결과 gate_40 < gate_30")

    if p_value < 0.05:
        print(f"p-value 값 {p_value}로 귀무가설 기각")
    elif p_value >= 0.05:
        print(f"p-value 값 {p_value}로 귀무가설 기각 실패")

    return 

In [82]:
# pooled proportion
d1_p_pool = pooled_proportion(d1_30, d1_40, d0_30, d0_40)

# 표준오차
d1_se = se_func(d1_p_pool, d0_30, d0_40)

# z-score
d1_z = z_score(r_d1_40, r_d1_30, d1_se)

print_result('d1', r_d1_40, r_d1_30, d1_p_pool, d1_se, d1_z)

group30 d1 retention: 0.4482, group40 d1 retention: 0.4423
pooled p: 0.4452
SE: 0.003310
z-score: -1.7841
가설 방향과 반대 결과 gate_40 < gate_30
p-value 값 0.9627951723515404로 귀무가설 기각 실패


In [83]:
# pooled proportion
d7_p_pool = pooled_proportion(d7_30, d7_40, d0_30, d0_40)

# 표준오차
d7_se = se_func(d7_p_pool, d0_30, d0_40)

# z-score
d7_z = z_score(r_d7_40, r_d7_30, d7_se)

print_result('d7', r_d7_40, r_d7_30, d7_p_pool, d7_se, d7_z)

group30 d7 retention: 0.1902, group40 d7 retention: 0.1820
pooled p: 0.1861
SE: 0.002592
z-score: -3.1644
가설 방향과 반대 결과 gate_40 < gate_30
p-value 값 0.9992228750121929로 귀무가설 기각 실패


---
## 결론
> - D1 retention 귀무가설 기각 실패로 gate_40의 retention은 gate_30보다 통계적으로 유의하게 높다고 판단할 수 없다.
> - D7 retention 귀무가설 기각 실패로 gate_40의 retention은 gate_30보다 통계적으로 유의하게 높다고 판단할 수 없다.

=> 따라서 gate 이동은 retention에 통계적으로 유의한 영향을 미친다고 결론을 내릴 충분한 근거가 없다.